In [1]:
# ================================================================
# PM2.5 Level Prediction using CNN (PyTorch)
# with SMOTE + Tomek Links (class 2,3 only) on train_split.csv
# ================================================================
import pandas as pd
import numpy as np
from collections import Counter

from sklearn.preprocessing import label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

from imblearn.combine import SMOTETomek

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [2]:
# ---------------- 1) Load dataset ----------------
train_df = pd.read_csv("train_split.csv")
test_df = pd.read_csv("test_split.csv")

X_train = train_df.drop(columns=["PM2.5_level"]).values.astype(np.float32)
y_train = train_df["PM2.5_level"].values.astype(np.int64) - 1

X_test = test_df.drop(columns=["PM2.5_level"]).values.astype(np.float32)
y_test = test_df["PM2.5_level"].values.astype(np.int64) - 1

print("Train size:", X_train.shape, " Test size:", X_test.shape)

Train size: (329623, 12)  Test size: (82406, 12)


In [3]:
# ---------------- 2) SMOTE + Tomek Links (ทุกคลาสเท่ากับคลาสที่มากที่สุด) ----------------
from imblearn.combine import SMOTETomek
from collections import Counter

# นับจำนวนคลาสใน train (หลัง shift แล้ว: 0..4)
before = Counter(y_train)
print("Before SMOTE+Tomek (train):", before)

# หา "จำนวน" ของคลาสที่มากที่สุดใน train
max_class_count = max(before.values())

# กำหนดให้ทุกคลาส oversample จนเท่ากับคลาสใหญ่สุด
sampling_strategy = {cls: max_class_count for cls in before.keys()}

# ทำ SMOTE + Tomek Links ให้ทุกคลาสเท่ากัน
smt = SMOTETomek(
    sampling_strategy=sampling_strategy,
    random_state=42
)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
after = Counter(y_train_res)

print("After SMOTE+Tomek (train):", after)


Before SMOTE+Tomek (train): Counter({np.int64(4): 128801, np.int64(3): 73062, np.int64(0): 64314, np.int64(1): 33504, np.int64(2): 29942})
After SMOTE+Tomek (train): Counter({np.int64(2): 128075, np.int64(0): 128046, np.int64(1): 127906, np.int64(4): 126161, np.int64(3): 125779})


In [4]:
# ---------------- 3) CNN Model Definition ----------------
class SimpleCNN(nn.Module):
    def __init__(self, num_features, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=0)
        self.fc1 = nn.Linear(self._get_conv_output(num_features), 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.5)

    def _get_conv_output(self, num_features):
        dummy = torch.zeros(1, 1, num_features)
        out = self.pool(F.relu(self.conv1(dummy)))
        out = self.pool(F.relu(self.conv2(out)))
        return out.view(1, -1).size(1)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [5]:
# ---------------- 4) Utility Function ----------------
def train_one_fold(model, optimizer, criterion, loader, device):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()

In [6]:
# ---------------- 5) 10-Fold Cross Validation ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(np.unique(y_train_res))

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

val_preds, val_probas, val_true = [], [], []

for train_idx, val_idx in skf.split(X_train_res, y_train_res):
    X_tr, X_val = X_train_res[train_idx], X_train_res[val_idx]
    y_tr, y_val = y_train_res[train_idx], y_train_res[val_idx]

    X_tr_t = torch.tensor(X_tr).unsqueeze(1)
    y_tr_t = torch.tensor(y_tr)
    X_val_t = torch.tensor(X_val).unsqueeze(1)
    y_val_t = torch.tensor(y_val)

    train_ds = TensorDataset(X_tr_t, y_tr_t)
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

    model_cv = SimpleCNN(X_tr.shape[1], num_classes).to(device)
    optimizer = torch.optim.Adam(model_cv.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(5):
        train_one_fold(model_cv, optimizer, criterion, train_loader, device)

    model_cv.eval()
    with torch.no_grad():
        out_val = model_cv(X_val_t.to(device))
        y_val_pred = out_val.argmax(dim=1).cpu().numpy()
        y_val_proba = F.softmax(out_val, dim=1).cpu().numpy()

    val_preds.extend(y_val_pred)
    val_probas.extend(y_val_proba)
    val_true.extend(y_val)

# Metrics
acc_val = accuracy_score(val_true, val_preds)
p_val, r_val, f1_val, _ = precision_recall_fscore_support(val_true, val_preds, average="weighted")
roc_val = roc_auc_score(
    label_binarize(val_true, classes=np.unique(y_train_res)),
    np.array(val_probas),
    multi_class="ovr",
    average="macro"
)

validation_table = pd.DataFrame([{
    "Accuracy": acc_val,
    "Precision": p_val,
    "Recall": r_val,
    "F1-score": f1_val,
    "ROC-AUC": roc_val
}])
print("\nValidation Table:\n", validation_table)


Validation Table:
    Accuracy  Precision    Recall  F1-score   ROC-AUC
0  0.685007   0.686119  0.685007  0.685332  0.918388


In [7]:
# ---------------- 6) Train Full Model & Evaluate on Test ----------------
X_train_res_t = torch.tensor(X_train_res).unsqueeze(1)
y_train_res_t = torch.tensor(y_train_res)
train_ds_full = TensorDataset(X_train_res_t, y_train_res_t)
train_loader_full = DataLoader(train_ds_full, batch_size=256, shuffle=True)

cnn_model = SimpleCNN(X_train_res.shape[1], num_classes).to(device)
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    train_one_fold(cnn_model, optimizer, criterion, train_loader_full, device)

cnn_model.eval()
X_test_t = torch.tensor(X_test).unsqueeze(1).to(device)
with torch.no_grad():
    out_test = cnn_model(X_test_t)
    y_test_pred = out_test.argmax(dim=1).cpu().numpy()
    y_test_proba = F.softmax(out_test, dim=1).cpu().numpy()

acc_test = accuracy_score(y_test, y_test_pred)
p_test, r_test, f1_test, _ = precision_recall_fscore_support(y_test, y_test_pred, average="weighted")
roc_test = roc_auc_score(
    label_binarize(y_test, classes=np.unique(y_train_res)),
    y_test_proba,
    multi_class="ovr",
    average="macro"
)

test_table = pd.DataFrame([{
    "Accuracy": acc_test,
    "Precision": p_test,
    "Recall": r_test,
    "F1-score": f1_test,
    "ROC-AUC": roc_test
}])
print("\nTest Table:\n", test_table)


Test Table:
    Accuracy  Precision    Recall  F1-score   ROC-AUC
0  0.726003   0.752985  0.726003  0.736792  0.929034


In [8]:
# ---------------- 7) Save Results ----------------
validation_table.to_csv("TomekSMOTE_validation_metrics_cnn.csv", index=False)
test_table.to_csv("TomekSMOTE_test_metrics_cnn.csv", index=False)

proba_df = pd.DataFrame(y_test_proba, columns=[f"prob_{c}" for c in np.unique(y_train_res)])
proba_df.insert(0, "true", y_test)
proba_df.to_csv("TomekSMOTE_proba_test_cnn.csv", index=False)

print("\nSaved: TomekSMOTE_validation_metrics_cnn.csv, TomekSMOTE_test_metrics_cnn.csv, TomekSMOTE_proba_test_cnn.csv")


Saved: TomekSMOTE_validation_metrics_cnn.csv, TomekSMOTE_test_metrics_cnn.csv, TomekSMOTE_proba_test_cnn.csv
